In [48]:
import os 
import ssl
import urllib.request

ssl._create_default_https_context = ssl._create_unverified_context

ocorrencias = [2025, 2026]
pasta = "sample_data"

os.makedirs(pasta, exist_ok=True)

for ano in ocorrencias:
    url = f"https://servicos.dpf.gov.br/dadosabertos/SINARM_CSV/OCORRENCIAS/OCORRENCIAS_{ano}.csv"
    caminho_arquivo = f"{pasta}/{ano}.csv"

    if not os.path.exists(caminho_arquivo):
        print(f"Arquivo {ano} ainda não criado. \n Buscando...")
        urllib.request.urlretrieve(url, caminho_arquivo)
    else:
        print("Arquivo já existe")
    

Arquivo já existe
Arquivo já existe


In [49]:
import pandas as pd

dataframe_listas = []

for ano in ocorrencias: 
    caminho_arquivo = f"{pasta}/{ano}.csv"

    if not os.path.exists(caminho_arquivo):
        print(f"Nao foi encontrado arquivo para o caminho dado: {caminho_arquivo}")
        continue

    dataframe_temp = pd.read_csv(caminho_arquivo, encoding="latin1", sep=";")
    dataframe_temp['ANO_OCORRENCIA'] = ano
    dataframe_listas.append(dataframe_temp)

dataframe_completo = pd.concat(dataframe_listas, ignore_index=True)

def clean_data(df):
    df_limpo = df.copy()
    
    df_limpo.columns = df_limpo.columns.str.strip().str.lower().str.replace(' ', '_')
    
    df_limpo = df_limpo.dropna(how='all')
    
    df_limpo = df_limpo.drop_duplicates()
    
    colunas_texto = df_limpo.select_dtypes(include=['object']).columns
    for col in colunas_texto:
        df_limpo[col] = df_limpo[col].str.strip()
        
    return df_limpo

print("Dataframe shape antes limpeza", dataframe_completo.shape)

# dataframe_completo = clean_data(dataframe_completo)

print("Dataframe shape pós limpeza", dataframe_completo.shape)

display(dataframe_completo.head())

Dataframe shape antes limpeza (85851, 10)
Dataframe shape pós limpeza (85851, 10)


,ANO_OCORRENCIA,MES_OCORRENCIA,UF,MUNICIPIO,ESPECIE_ARMA,MARCA_ARMA,CALIBRE_ARMA,TIPO_OCORRENCIA,MAIS_1000_MIL_HAB,TOTAL
0,2025,1,AC,ACRELÂNDIA,Espingarda ...,ROSSI (AMADEO ROSSI S.A.) ...,36 ...,Apreensão de Arma de Fogo ...,N,1
1,2025,1,AC,CRUZEIRO DO SUL,Espingarda ...,ROSSI (AMADEO ROSSI S.A.) ...,28 ...,Extravio/Perda de Arma de Fogo ...,N,1
2,2025,1,AC,FEIJÓ,Espingarda ...,CBC (COMPANHIA BRASILEIRA DE CARTUCHOS) ...,28 ...,Extravio/Perda de Arma de Fogo ...,N,1
3,2025,1,AC,RIO BRANCO,Espingarda ...,ROSSI (AMADEO ROSSI S.A.) ...,36/.22LR (duplo cano) ...,Extravio/Perda de Arma de Fogo ...,S,2
4,2025,1,AC,RIO BRANCO,Pistola ...,TAURUS ARMAS S.A. ...,.380 ACP ...,Roubo de Arma de Fogo ...,S,1


In [50]:
# dataframe_completo.to_csv(f"sample_data/dataset_limpo.csv", index=False, encoding='latin1', sep=';')
dataframe_completo.to_parquet(f"sample_data/dataset_limpo.parquet", index=False)